# Utilities

In [62]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [63]:
from Utils.Config import ExperimentResult
from pprint import pprint


json_path = Path("Cache/MetaData/trial_0034/experiment_summary.json")

with open(json_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

exp = ExperimentResult.from_jsonable(raw)

pprint(exp)

ExperimentResult(transform_hparams=TransformHyperParams(resize_size=256,
                                                        p=1.0,
                                                        prefix='resizecrop',
                                                        bilateral_d=11,
                                                        sigma_color=170,
                                                        sigma_space=75,
                                                        gaussian_k=11,
                                                        gaussian_sigma=2.0,
                                                        nlmeans_h=20,
                                                        template_window_size=11,
                                                        search_window_size=11,
                                                        gray_alpha=1.0,
                                                        grid_size=6,
                                                    

# 0. Config

In [64]:
ROOT = Path("Cache/MetaData")  # trial folders root
SAVE_DIR = Path("analysis_outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PERTURB_TO_FEATURE = {
    "localwarp": "shape",
    "patchshuffle": "shape",
    "patchrotation": "shape",
    "grayscale": "color",
    "bilateral": "texture",
    "bilateralfilter": "texture",
    "original": "clean",
    "none": "clean",
}


# 1. Utilities

In [65]:
def normalize_perturbation(name):
    name = str(name).lower().replace("_", "").replace("-", "")

    if name in {"original", "clean", "none"}:
        return "original"
    if "localwarp" in name:
        return "localwarp"
    if "patchshuffle" in name:
        return "patchshuffle"
    if "patchrotation" in name or "patchrotate" in name:
        return "patchrotation"
    if "gray" in name:
        return "grayscale"
    if "bilateral" in name:
        return "bilateral"

    return name


def parse_scenario_key(key):
    if "__" not in str(key):
        return None, None

    dataset, perturb = str(key).split("__", 1)
    return dataset, normalize_perturbation(perturb)


def make_scenario_key(dataset, perturbation):
    return f"{dataset}__{normalize_perturbation(perturbation)}"


def pick(d, *keys, default=np.nan):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return default


def build_dataset_meta(summary):
    return {
        ds["name"]: ds
        for ds in summary.get("data_config", {}).get("datasets", [])
    }


def has_validation_metrics(d):
    if not isinstance(d, dict):
        return False

    metric_keys = {
        "LV_ratio", "HFE_ratio", "ESSIM", "GC", "gc",
        "texture_score", "shape_score",
    }

    return any(k in d for k in metric_keys)


def extract_validation_metrics(obj):
    if not isinstance(obj, dict):
        return {}

    if has_validation_metrics(obj):
        return obj

    wrapper_keys = [
        "metrics",
        "scores",
        "result",
        "results",
        "validation",
        "validation_metrics",
        "perturbation_metrics",
    ]

    for key in wrapper_keys:
        if key in obj and isinstance(obj[key], dict):
            found = extract_validation_metrics(obj[key])
            if found:
                return found

    return {}


def find_validation_metrics(perturb_val, perturbation):
    """
    Find validation metrics from the actual JSON structure.

    Expected current case:
        perturbation_validation[perturbation] = {...}

    Also supports:
        perturbation_validation["results"][perturbation] = {...}
        perturbation_validation["records"][i]["perturbation"] = perturbation
        perturbation_validation[perturbation]["metrics"] = {...}
    """
    perturbation = normalize_perturbation(perturbation)

    if not perturb_val:
        return {}

    # Case 1: direct perturbation key
    if isinstance(perturb_val, dict):
        for key, value in perturb_val.items():
            if normalize_perturbation(key) == perturbation:
                found = extract_validation_metrics(value)
                if found:
                    return found

        # Case 2: common nested containers
        container_keys = [
            "results",
            "records",
            "items",
            "validations",
            "perturbations",
            "perturbation_results",
            "perturbation_validation",
            "validation_results",
        ]

        for key in container_keys:
            if key not in perturb_val:
                continue

            container = perturb_val[key]

            if isinstance(container, dict):
                for sub_key, value in container.items():
                    if normalize_perturbation(sub_key) == perturbation:
                        found = extract_validation_metrics(value)
                        if found:
                            return found

            if isinstance(container, list):
                for item in container:
                    if not isinstance(item, dict):
                        continue

                    item_perturb = pick(
                        item,
                        "perturbation",
                        "perturb",
                        "name",
                        "transform",
                        "transform_name",
                        default=None,
                    )

                    if item_perturb is not None and normalize_perturbation(item_perturb) == perturbation:
                        found = extract_validation_metrics(item)
                        if found:
                            return found

        # Case 3: recursive fallback
        for value in perturb_val.values():
            if isinstance(value, dict):
                found = find_validation_metrics(value, perturbation)
                if found:
                    return found

            if isinstance(value, list):
                for item in value:
                    if not isinstance(item, dict):
                        continue

                    item_perturb = pick(
                        item,
                        "perturbation",
                        "perturb",
                        "name",
                        "transform",
                        "transform_name",
                        default=None,
                    )

                    if item_perturb is not None and normalize_perturbation(item_perturb) == perturbation:
                        found = extract_validation_metrics(item)
                        if found:
                            return found

    if isinstance(perturb_val, list):
        for item in perturb_val:
            if not isinstance(item, dict):
                continue

            item_perturb = pick(
                item,
                "perturbation",
                "perturb",
                "name",
                "transform",
                "transform_name",
                default=None,
            )

            if item_perturb is not None and normalize_perturbation(item_perturb) == perturbation:
                found = extract_validation_metrics(item)
                if found:
                    return found

    return {}


def build_scenario_df(summary, json_path):
    trial_id = json_path.parent.name
    hparams = summary.get("transform_hparams", {})
    dataset_meta = build_dataset_meta(summary)

    rows = []

    for scenario in summary.get("scenarios", []):
        dataset = pick(scenario, "dataset_name", "dataset")
        perturbation = normalize_perturbation(
            pick(scenario, "perturbation", default="original")
        )
        scenario_key = make_scenario_key(dataset, perturbation)
        ds_info = dataset_meta.get(dataset, {})

        rows.append({
            "trial_id": trial_id,
            "trial_path": str(json_path),

            "scenario_key": scenario_key,
            "dataset": dataset,
            "perturbation": perturbation,

            "dataset_type": ds_info.get("dataset_type", np.nan),
            "domain_type": ds_info.get("domain_type", np.nan),
            "shift_type": ds_info.get("shift_type", np.nan),
            "num_classes": ds_info.get("num_classes", np.nan),

            "feature_type": PERTURB_TO_FEATURE.get(perturbation, "unknown"),

            "alpha": hparams.get("alpha_localwarp", np.nan),
            "sigma": hparams.get("sigma_localwarp", np.nan),
            "grid_size": hparams.get("grid_size", np.nan),
        })

    scenario_df = pd.DataFrame(rows)

    if scenario_df.empty:
        seen = set()

        for model_result in summary.get("model_results", {}).values():
            for raw_scenario_key in model_result.get("scenario_results", {}).keys():
                dataset, perturbation = parse_scenario_key(raw_scenario_key)

                if dataset is None:
                    continue

                perturbation = normalize_perturbation(perturbation)
                scenario_key = make_scenario_key(dataset, perturbation)

                if scenario_key in seen:
                    continue

                seen.add(scenario_key)
                ds_info = dataset_meta.get(dataset, {})

                rows.append({
                    "trial_id": trial_id,
                    "trial_path": str(json_path),

                    "scenario_key": scenario_key,
                    "dataset": dataset,
                    "perturbation": perturbation,

                    "dataset_type": ds_info.get("dataset_type", np.nan),
                    "domain_type": ds_info.get("domain_type", np.nan),
                    "shift_type": ds_info.get("shift_type", np.nan),
                    "num_classes": ds_info.get("num_classes", np.nan),

                    "feature_type": PERTURB_TO_FEATURE.get(perturbation, "unknown"),

                    "alpha": hparams.get("alpha_localwarp", np.nan),
                    "sigma": hparams.get("sigma_localwarp", np.nan),
                    "grid_size": hparams.get("grid_size", np.nan),
                })

        scenario_df = pd.DataFrame(rows)

    return scenario_df


def build_validation_df(summary, json_path, scenario_df):
    """
    One row = trial_id + perturbation

    validation is currently measured on ImageNet-1K by default.
    dataset column is kept for future dataset-specific validation.
    """
    trial_id = json_path.parent.name
    perturb_val = summary.get("perturbation_validation") or {}
    hparams = summary.get("transform_hparams", {})

    alpha = hparams.get("alpha_localwarp", np.nan)
    sigma = hparams.get("sigma_localwarp", np.nan)
    grid_size = hparams.get("grid_size", np.nan)

    rows = []

    perturbations = (
        scenario_df["perturbation"]
        .dropna()
        .map(normalize_perturbation)
        .unique()
        .tolist()
        if "perturbation" in scenario_df.columns
        else []
    )

    for perturbation in perturbations:
        metrics = find_validation_metrics(perturb_val, perturbation)

        rows.append({
            "trial_id": trial_id,
            "trial_path": str(json_path),

            "dataset": "imagenet",

            "perturbation": perturbation,
            "controlled_perturbation": perturbation if perturbation != "original" else "none",

            "alpha": alpha,
            "sigma": sigma,
            "grid_size": grid_size,

            "LV_ratio": metrics.get("LV_ratio", np.nan),
            "HFE_ratio": metrics.get("HFE_ratio", np.nan),
            "ESSIM": metrics.get("ESSIM", np.nan),
            "GC": metrics.get("GC", metrics.get("gc", np.nan)),
            "texture_score": metrics.get("texture_score", np.nan),
            "shape_score": metrics.get("shape_score", np.nan),
        })

    return pd.DataFrame(rows)


def build_result_df(summary, json_path):
    trial_id = json_path.parent.name
    rows = []

    for model_key, model_result in summary.get("model_results", {}).items():
        model_name = model_result.get("model_name", model_key)
        pretrained = model_result.get("pretrained_weight", np.nan)

        for raw_scenario_key, rec in model_result.get("scenario_results", {}).items():
            key_dataset, key_perturbation = parse_scenario_key(raw_scenario_key)

            dataset = pick(rec, "dataset_name", "dataset", default=key_dataset)
            perturbation = normalize_perturbation(
                pick(rec, "perturbation", default=key_perturbation)
            )
            scenario_key = make_scenario_key(dataset, perturbation)

            rows.append({
                "trial_id": trial_id,
                "trial_path": str(json_path),

                "model_key": model_key,
                "model": model_name,
                "pretrained": pretrained,

                "scenario_key": scenario_key,
                "raw_scenario_key": raw_scenario_key,
                "dataset": dataset,
                "perturbation": perturbation,

                "accuracy": pick(rec, "accuracy", "perturbed_accuracy"),
                "relative_accuracy": pick(
                    rec,
                    "relative_accuracy",
                    "relative_accuracy_score",
                    "rel_accuracy",
                    "rel_acc",
                ),
                "js": pick(rec, "js", "js_divergence"),
                "cka": pick(rec, "cka"),
                "drop_same": pick(
                    rec,
                    "drop_same",
                    "accuracy_drop_vs_same_dataset_clean",
                ),
                "drop_id": pick(
                    rec,
                    "drop_id",
                    "accuracy_drop_vs_id_clean",
                ),
            })

    return pd.DataFrame(rows)


def parse_one_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    scenario_df = build_scenario_df(summary, json_path)
    validation_df = build_validation_df(summary, json_path, scenario_df)
    result_df = build_result_df(summary, json_path)

    master_df = (
        result_df
        .merge(
            scenario_df.drop(columns=["trial_path"], errors="ignore"),
            on=["trial_id", "scenario_key", "dataset", "perturbation"],
            how="left",
        )
        .merge(
            validation_df.drop(columns=["trial_path"], errors="ignore"),
            on=["trial_id", "perturbation"],
            how="left",
        )
    )

    return {
        "scenario_df": scenario_df,
        "validation_df": validation_df,
        "result_df": result_df,
        "master_df": master_df,
    }


def make_master_dataframe(
    root_dir=Path("Cache/MetaData"),
    save_dir=Path("analysis_outputs"),
    save_csv=True,
):
    save_dir.mkdir(parents=True, exist_ok=True)

    scenario_list = []
    validation_list = []
    result_list = []
    master_list = []

    for json_path in root_dir.rglob("experiment_summary.json"):
        try:
            parsed = parse_one_json(json_path)

            scenario_list.append(parsed["scenario_df"])
            validation_list.append(parsed["validation_df"])
            result_list.append(parsed["result_df"])
            master_list.append(parsed["master_df"])

        except Exception as e:
            print(f"[WARN] {json_path}: {e}")

    scenario_df = pd.concat(scenario_list, ignore_index=True) if scenario_list else pd.DataFrame()
    validation_df = pd.concat(validation_list, ignore_index=True) if validation_list else pd.DataFrame()
    result_df = pd.concat(result_list, ignore_index=True) if result_list else pd.DataFrame()
    master_df = pd.concat(master_list, ignore_index=True) if master_list else pd.DataFrame()

    # Columns to rename from merge suffixes
    rename_map = {
        "dataset_x": "dataset",
        "alpha_x": "alpha",
        "sigma_x": "sigma",
        "grid_size_x": "grid_size",
    }

    # Columns to drop
    drop_cols = [
        # duplicated by merge
        "dataset_y",
        "alpha_y",
        "sigma_y",
        "grid_size_y",

        # implementation-only metadata
        "dataset_type",
        "raw_scenario_key",

        # optional: keep only if you need debugging
        # "trial_path",
    ]


    master_df = master_df.rename(columns=rename_map).drop(columns=drop_cols, errors="ignore")

    num_cols = [
        "num_classes", "alpha", "sigma", "grid_size",
        "accuracy", "relative_accuracy", "js", "cka",
        "drop_same", "drop_id",
        "LV_ratio", "HFE_ratio", "ESSIM", "GC",
        "texture_score", "shape_score",
    ]

    for df in [scenario_df, validation_df, result_df, master_df]:
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    if "relative_accuracy" in master_df.columns:
        master_df["feature_reliance"] = 1.0 - master_df["relative_accuracy"]

    if "perturbation" in master_df.columns:
        master_df["is_localwarp"] = master_df["perturbation"].eq("localwarp")

    if save_csv:
        scenario_df.to_csv(save_dir / "scenario.csv", index=False)
        validation_df.to_csv(save_dir / "validation.csv", index=False)
        result_df.to_csv(save_dir / "result.csv", index=False)
        master_df.to_csv(save_dir / "master.csv", index=False)

    print("scenario_df:", scenario_df.shape)
    print("validation_df:", validation_df.shape)
    print("result_df:", result_df.shape)
    print("master_df:", master_df.shape)

    if not validation_df.empty:
        print("\nvalidation missing ratio:")
        display(validation_df[[
            "LV_ratio", "HFE_ratio", "ESSIM", "GC",
            "texture_score", "shape_score"
        ]].isna().mean())

    if not master_df.empty:
        display(master_df.head())

    return {
        "scenario_df": scenario_df,
        "validation_df": validation_df,
        "result_df": result_df,
        "master_df": master_df,
    }

In [66]:
result_df = make_master_dataframe(root_dir=ROOT, save_dir=SAVE_DIR)
master_df = result_df["master_df"]
validation_df = result_df["validation_df"]
scenario_df = result_df["scenario_df"]


master_df.head()

scenario_df: (630, 13)
validation_df: (210, 14)
result_df: (1260, 15)
master_df: (1260, 30)

validation missing ratio:


LV_ratio         0.166667
HFE_ratio        0.166667
ESSIM            0.166667
GC               0.166667
texture_score    0.166667
shape_score      0.166667
dtype: float64

,trial_id,trial_path,model_key,model,pretrained,scenario_key,dataset,perturbation,accuracy,relative_accuracy,...,grid_size,controlled_perturbation,LV_ratio,HFE_ratio,ESSIM,GC,texture_score,shape_score,feature_reliance,is_localwarp
0,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__original,imagenet,original,0.79298,NaN,...,6,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__grayscale,imagenet,grayscale,0.70596,0.890124,...,6,grayscale,0.810447,0.795543,0.257438,0.033165,0.802995,0.145302,0.109876,False
2,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__bilateral,imagenet,bilateral,0.61658,0.777267,...,6,bilateral,0.451907,0.389136,0.251413,0.052750,0.420522,0.152081,0.222733,False
3,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__patchshuffle,imagenet,patchshuffle,0.56990,0.718326,...,6,patchshuffle,0.975195,0.966811,0.158148,0.002505,0.971003,0.080327,0.281674,False
4,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__patchrotation,imagenet,patchrotation,0.35850,0.451400,...,6,patchrotation,0.880642,0.860810,0.219983,0.013713,0.870726,0.116848,0.548600,False


,trial_id,trial_path,model_key,model,pretrained,scenario_key,dataset,perturbation,accuracy,relative_accuracy,...,grid_size,controlled_perturbation,LV_ratio,HFE_ratio,ESSIM,GC,texture_score,shape_score,feature_reliance,is_localwarp
0,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__original,imagenet,original,0.79298,NaN,...,6,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__grayscale,imagenet,grayscale,0.70596,0.890124,...,6,grayscale,0.810447,0.795543,0.257438,0.033165,0.802995,0.145302,0.109876,False
2,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__bilateral,imagenet,bilateral,0.61658,0.777267,...,6,bilateral,0.451907,0.389136,0.251413,0.052750,0.420522,0.152081,0.222733,False
3,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__patchshuffle,imagenet,patchshuffle,0.56990,0.718326,...,6,patchshuffle,0.975195,0.966811,0.158148,0.002505,0.971003,0.080327,0.281674,False
4,trial_0000,Cache\MetaData\trial_0000\experiment_summary.json,resnet50__in1k,resnet50,in1k,imagenet__patchrotation,imagenet,patchrotation,0.35850,0.451400,...,6,patchrotation,0.880642,0.860810,0.219983,0.013713,0.870726,0.116848,0.548600,False


# 2. Sanity Check

In [67]:
print(master_df.columns.tolist())

['trial_id', 'trial_path', 'model_key', 'model', 'pretrained', 'scenario_key', 'dataset', 'perturbation', 'accuracy', 'relative_accuracy', 'js', 'cka', 'drop_same', 'drop_id', 'domain_type', 'shift_type', 'num_classes', 'feature_type', 'alpha', 'sigma', 'grid_size', 'controlled_perturbation', 'LV_ratio', 'HFE_ratio', 'ESSIM', 'GC', 'texture_score', 'shape_score', 'feature_reliance', 'is_localwarp']


In [71]:
print(master_df.shape)
print(master_df.columns)
print(master_df.isna().sum())

display(master_df[[
    "dataset",
    "model_key",
    "perturbation",
    "accuracy",
    "drop_same",
    "drop_id"
]].head(20))

(1260, 30)
Index(['trial_id', 'trial_path', 'model_key', 'model', 'pretrained',
       'scenario_key', 'dataset', 'perturbation', 'accuracy',
       'relative_accuracy', 'js', 'cka', 'drop_same', 'drop_id', 'domain_type',
       'shift_type', 'num_classes', 'feature_type', 'alpha', 'sigma',
       'grid_size', 'controlled_perturbation', 'LV_ratio', 'HFE_ratio',
       'ESSIM', 'GC', 'texture_score', 'shape_score', 'feature_reliance',
       'is_localwarp'],
      dtype='object')
trial_id                     0
trial_path                   0
model_key                    0
model                        0
pretrained                   0
scenario_key                 0
dataset                      0
perturbation                 0
accuracy                     0
relative_accuracy          210
js                         210
cka                        210
drop_same                  210
drop_id                      0
domain_type                  0
shift_type                 840
num_classes         

,dataset,model_key,perturbation,accuracy,drop_same,drop_id
0,imagenet,resnet50__in1k,original,0.792980,NaN,0.000000
1,imagenet,resnet50__in1k,grayscale,0.705960,0.087020,0.087020
2,imagenet,resnet50__in1k,bilateral,0.616580,0.176400,0.176400
3,imagenet,resnet50__in1k,patchshuffle,0.569900,0.223080,0.223080
4,imagenet,resnet50__in1k,patchrotation,0.358500,0.434480,0.434480
5,imagenet,resnet50__in1k,localwarp,0.792980,0.000000,0.000000
6,imagenet_200,resnet50__in1k,original,0.931600,NaN,0.000000
7,imagenet_200,resnet50__in1k,grayscale,0.872200,0.059400,0.059400
8,imagenet_200,resnet50__in1k,bilateral,0.811600,0.120000,0.120000
9,imagenet_200,resnet50__in1k,patchshuffle,0.780100,0.151500,0.151500


In [72]:
master_df.groupby(["dataset", "model_key", "perturbation"]).size()

dataset       model_key           perturbation 
imagenet      resnet50__in1k      bilateral        35
                                  grayscale        35
                                  localwarp        35
                                  original         35
                                  patchrotation    35
                                  patchshuffle     35
              vit-b__augreg_in1k  bilateral        35
                                  grayscale        35
                                  localwarp        35
                                  original         35
                                  patchrotation    35
                                  patchshuffle     35
imagenet_200  resnet50__in1k      bilateral        35
                                  grayscale        35
                                  localwarp        35
                                  original         35
                                  patchrotation    35
                                  

# 3. Analysis

## 3-1. Perturbation Validate Check

In [74]:
fixed_df = master_df[master_df["is_localwarp"] == False]

fixed_summary = (
    fixed_df
    .groupby(["domain_type", "perturbation", "feature_type"])[
        ["texture_score", "shape_score", "accuracy", "drop_same", "drop_id", "js", "cka"]
    ]
    .mean()
    .reset_index()
)

display(fixed_summary)

,domain_type,perturbation,feature_type,texture_score,shape_score,accuracy,drop_same,drop_id,js,cka
0,id,bilateral,texture,0.420522,0.152081,0.729700,0.132770,0.132770,0.155549,0.719627
1,id,grayscale,color,0.802995,0.145302,0.775575,0.086895,0.086895,0.114381,0.766769
2,id,original,clean,NaN,NaN,0.862470,NaN,0.000000,NaN,NaN
3,id,patchrotation,shape,0.870933,0.116735,0.519395,0.343075,0.343075,0.341161,0.509459
4,id,patchshuffle,shape,0.971042,0.080373,0.644660,0.217810,0.217810,0.227161,0.629495
5,natural_ood,bilateral,texture,0.420522,0.152081,0.380650,0.019250,0.552000,0.238730,0.697604
6,natural_ood,grayscale,color,0.802995,0.145302,0.361800,0.038100,0.570850,0.205504,0.742477
7,natural_ood,original,clean,NaN,NaN,0.399900,NaN,0.532750,NaN,NaN
8,natural_ood,patchrotation,shape,0.870933,0.116735,0.102933,0.296967,0.829717,0.432515,0.470431
9,natural_ood,patchshuffle,shape,0.971042,0.080373,0.149417,0.250483,0.783233,0.382730,0.528603


In [ ]:
lw_df = master_df[master_df["is_localwarp"] == True]
lw_summary = (
    lw_df
    .groupby(["domain_type", "alpha", "sigma"])[
        ["texture_score", "shape_score", "accuracy", "drop_same", "drop_id", "js", "cka"]
    ]
    .mean()
    .reset_index()
)